# Traffic Analysis Notebook
Explorative Analyse auf Basis der zusammengeführten Verkehrsdaten.

In [11]:
import pandas as pd

df = pd.read_csv('../data/processed/traffic_data_combined.csv')
df.head(200)

,Datum,Standort,Richtung,Spur,Klasse.ID,Klasse.Text,Intervall,Anzahl,Status,year
0,2020-01-21T00:00+0100,Zch_Rosengartenbrücke,Hardbrücke,0,0,Unbekannt,h1,0,provisorisch,2020
1,2020-01-21T00:00+0100,Zch_Rosengartenbrücke,Hardbrücke,0,1,Bus,h1,0,provisorisch,2020
2,2020-01-21T00:00+0100,Zch_Rosengartenbrücke,Hardbrücke,0,2,Motorrad,h1,0,provisorisch,2020
3,2020-01-21T00:00+0100,Zch_Rosengartenbrücke,Hardbrücke,0,3,Personenwagen,h1,0,provisorisch,2020
4,2020-01-21T00:00+0100,Zch_Rosengartenbrücke,Hardbrücke,0,4,Personenwagen mit Anhänger,h1,0,provisorisch,2020
...,...,...,...,...,...,...,...,...,...,...
195,2020-01-21T05:00+0100,Zch_Rosengartenbrücke,Hardbrücke,1,3,Personenwagen,h1,391,provisorisch,2020
196,2020-01-21T05:00+0100,Zch_Rosengartenbrücke,Hardbrücke,1,4,Personenwagen mit Anhänger,h1,1,provisorisch,2020
197,2020-01-21T05:00+0100,Zch_Rosengartenbrücke,Hardbrücke,1,5,Lieferwagen,h1,37,provisorisch,2020
198,2020-01-21T05:00+0100,Zch_Rosengartenbrücke,Hardbrücke,1,6,Lieferwagen mit Anhänger,h1,2,provisorisch,2020


In [7]:
df.columns

Index(['Datum', 'Standort', 'Richtung', 'Spur', 'Klasse.ID', 'Klasse.Text',
       'Intervall', 'Anzahl', 'Status', 'year'],
      dtype='str')

In [8]:
df.shape

(4551876, 10)

In [9]:
# Mini-Analyse: sehr einfach und direkt
print('Zeilen, Spalten:', df.shape)
print('\nDatentypen:')
print(df.dtypes)
print('\nJahre in den Daten:')
print(sorted(df['year'].dropna().unique().tolist()))
print('\nTop 10 Klassen nach Gesamtanzahl:')
print(
    df.groupby('Klasse.Text', dropna=False)['Anzahl']
      .sum()
      .sort_values(ascending=False)
      .head(10)
)
print('\nFehlende Werte pro Spalte:')
print(df.isna().sum())

Zeilen, Spalten: (4551876, 10)

Datentypen:
Datum            str
Standort         str
Richtung         str
Spur           int64
Klasse.ID      int64
Klasse.Text      str
Intervall        str
Anzahl         int64
Status           str
year           int64
dtype: object

Jahre in den Daten:
[2020, 2021, 2022, 2023, 2024, 2025, 2026]

Top 10 Klassen nach Gesamtanzahl:
Klasse.Text
Personenwagen                 108225278
Lieferwagen                    11005103
Motorrad                        5668623
Lastwagen                       2354008
Trolleybus                      1427482
Sattelzug                        660927
Personenwagen mit Anhänger       564276
Lieferwagen mit Anhänger         297906
Bus                              222550
Lastenzug                        206523
Name: Anzahl, dtype: int64

Fehlende Werte pro Spalte:
Datum          0
Standort       0
Richtung       0
Spur           0
Klasse.ID      0
Klasse.Text    0
Intervall      0
Anzahl         0
Status         0
year         

In [19]:
# Distinct Standort prüfen
distinct_standorte = sorted(df['Standort'].dropna().unique().tolist())

print('Anzahl unterschiedlicher Standorte:', len(distinct_standorte))
print('Werte:', distinct_standorte)

# Optional: Verteilung je Standort
print('\nZeilen pro Standort:')
print(df['Standort'].value_counts(dropna=False))

Anzahl unterschiedlicher Standorte: 1
Werte: ['Zch_Rosengartenbrücke']

Zeilen pro Standort:
Standort
Zch_Rosengartenbrücke    4551876
Name: count, dtype: int64


In [20]:
# Kleine Transformation für Power BI
df_powerbi = df.copy()

# Datum mit Uhrzeit behalten, aber ohne Zeitzonen-Information speichern
df_powerbi['Datum'] = (
    pd.to_datetime(df_powerbi['Datum'], errors='coerce', utc=True)
      .dt.tz_convert('Europe/Zurich')
      .dt.tz_localize(None)
      .dt.strftime('%Y-%m-%d %H:%M:%S')
)

# Nicht benötigte Spalten entfernen
df_powerbi = df_powerbi.drop(columns=['year', 'Intervall', 'Status', 'Standort'], errors='ignore')

output_path = '../data/processed/traffic_data_powerbi.csv'
df_powerbi.to_csv(output_path, index=False)

print(f'Datei gespeichert: {output_path}')
print('Spalten:', df_powerbi.columns.tolist())
df_powerbi.head()

Datei gespeichert: ../data/processed/traffic_data_powerbi.csv
Spalten: ['Datum', 'Richtung', 'Spur', 'Klasse.ID', 'Klasse.Text', 'Anzahl']


,Datum,Richtung,Spur,Klasse.ID,Klasse.Text,Anzahl
0,2020-01-21 00:00:00,Hardbrücke,0,0,Unbekannt,0
1,2020-01-21 00:00:00,Hardbrücke,0,1,Bus,0
2,2020-01-21 00:00:00,Hardbrücke,0,2,Motorrad,0
3,2020-01-21 00:00:00,Hardbrücke,0,3,Personenwagen,0
4,2020-01-21 00:00:00,Hardbrücke,0,4,Personenwagen mit Anhänger,0
